In [1]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Some global variables

In [4]:
filter_start = "2025-10-01"
filter_end = "2026-10-01"

period_start = "2023-10-01"
period_end = "2026-10-01"

Extracting stocks

In [10]:
# EXTRACTING TICKERS. MAKE SURE TO CHANGE FILE NAME FOR ACTUAL SUBMISSION
ticker_file = "Tickers_Example.csv"
raw_tickers = pd.read_csv(ticker_file, header=None, names=["Ticker"]) 
tickers = raw_tickers["Ticker"].drop_duplicates().tolist()  
# deletes the repeated tickers

# my updates here：
# using .get() to avoid keyerror when a field is missing
# move the volume and close to a seperate cells
# add sector info for the 40% sector weight restriction

stock_rows = []

for tick in tickers:
    t = yf.Ticker(tick)
    information = t.get_info()

    # Check if there is data in here (i.e. not a delisted or invalid ticker)
    if t.history(period='max').empty:
        print(tick, "doesn't exist")
        continue
    
    # Check if it's a CAD or USD stock traded in CA or US market
    if information.get("market") not in ["ca_market", "us_market"]:
        print(tick, "wrong market")
        continue

    # Check if it's a stock and not something like a bond or index fund
    if information.get("quoteType") != "EQUITY":
        print(tick, "not stock")
        continue

    # check currency
    if information.get("currency") not in ["CAD", "USD"]:
        print(tick, "currency wrong")
        continue

    # check sector info
    if not information.get("sector"):
        print(tick, "sector missing")
        continue

    # save stock info for later processing
    stock_rows.append({
            "Ticker": tick,
            "Currency": information.get("currency"),
            "Sector": information.get("sector")
        })

stock_info = pd.DataFrame(stock_rows).set_index("Ticker") 
# set ticker as index
listed_tickers = stock_info.index.tolist()
# save the list for filter
display(stock_info.head())

$AGN: possibly delisted; no timezone found


AGN doesn't exist


$BK: possibly delisted; no price data found  (1d 1927-12-15 -> 2026-11-20)


BK doesn't exist


$CELG: possibly delisted; no timezone found


CELG doesn't exist


$MON: possibly delisted; no timezone found


MON doesn't exist


$RTN: possibly delisted; no timezone found


RTN doesn't exist


,Currency,Sector
Ticker,,
AAPL,USD,Technology
ABBV,USD,Healthcare
ABT,USD,Healthcare
ACN,USD,Technology
AIG,USD,Financial Services


In [ ]:
# Filters out the first and last trading day of each month
# idx = close_prices.index.to_series()
# date_series_first=idx.groupby([idx.dt.year, idx.dt.month]).first()
# date_series_last=idx.groupby([idx.dt.year, idx.dt.month]).last()
# monthly_first = []
# monthly_last = []
# for i in date_series_first:
#     monthly_first.append(i)
# for i in date_series_last:
#     monthly_last.append(i)

# for i in range(len(monthly_first)):
#     start_ind = close_prices.index.get_loc(monthly_first[i])
#     end_ind = close_prices.index.get_loc(monthly_last[i])

#     if (end_ind - start_ind + 1) < 18:
#         close_prices.drop(close_prices.loc[monthly_first[i]:monthly_last[i]].index)

# I suggest move volume check and valid trading dasy for each stock separately


In [27]:
# download historical data
history = yf.download(
    listed_tickers + ["CAD=X"],
    start=period_start,
    end=period_end,
    threads=False,
)

# download close, volume and foreign exchange info
close_prices = history["Close"][listed_tickers]
# get closing price info
volume_data = history["Volume"][listed_tickers]
# get volume info
cad_per_usd = history["Close"]["CAD=X"]
# get USD to CAD exchange info

[*********************100%***********************]  36 of 36 completed


In [28]:
# create a seperate volume data for filter
volume_data = volume_data.loc[(volume_data.index >= filter_start) & (volume_data.index < filter_end)]
print(f"volume screening period: {volume_data.index.min()}, {volume_data.index.max()}")

# updates:
# average daily trading volume = total volume traded over a period (volume) / # of trading dasy in the period
# since months with fewer than 18 valid trading days may affect the average daily trading volume
# 1. filter out months with fewer than 18 valid trading days before calculating liquidity
# 2. use only eligible months to calculate liquidity and determine eligible_tickers

eligible_tickers = []
average_volume = {} 

for ticker in listed_tickers:
    volume = volume_data[ticker].dropna()  
    # delete missed value
    volume_days = []

    for month, monthly_volume in volume.groupby(volume.index.to_period("M")): 
        if len(monthly_volume) >= 18:  
            volume_days.extend(monthly_volume)  

    average_volume[ticker] = pd.Series(volume_days).mean()  
    # transfer list to series to calculate average trading volume
    if average_volume[ticker] >= 5_000:
        eligible_tickers.append(ticker)

stock_info["Average Daily Volume"] = pd.Series(average_volume)  
print(f"{len(eligible_tickers)} eligible stocks.")

display(stock_info["Average Daily Volume"].head())

volume screening period: 2025-10-01 00:00:00, 2026-09-30 00:00:00
35 eligible stocks.


Ticker
AAPL    4.828203e+07
ABBV    6.281745e+06
ABT     1.008423e+07
ACN     6.116120e+06
AIG     4.324195e+06
Name: Average Daily Volume, dtype: float64

In [29]:
# updates:
# make sure all stocks are using the same currency

# convert prices to CAD
cad_prices = close_prices[eligible_tickers].copy()
for ticker in eligible_tickers:
    if stock_info.loc[ticker, "Currency"] == "USD":
        # if currency is USD, then change to CAD
        cad_prices[ticker] = cad_prices[ticker] * cad_per_usd
        # match closing price and exchange rate by date

cad_prices = cad_prices.dropna()  
# delete dates which missing info

In [32]:
# updates
# approach to calculate mean and std which gives equal weight to all data
# it is used for the low-risk portfolio selection only so far
daily_returns = cad_prices.pct_change().iloc[1:]  
# skip day1 since it is nan
mean_returns = daily_returns.mean()  
# calculate the mean daily returns of each stock
daily_std = daily_returns.std(ddof=1)  
# calculate the standard deviation of daily returns of stocks

stock_stats = stock_info.loc[eligible_tickers].copy()
stock_stats["Mean Return (%)"] = 100 * mean_returns
stock_stats["Daily Std (%)"] = 100 * daily_std

# demo
display(daily_returns.iloc[:5, :5])
display(stock_stats.round(4))

Ticker,AAPL,ABBV,ABT,ACN,AIG
Date,,,,,
2023-10-03,-0.000270,0.001986,0.008190,0.000923,-0.016099
2023-10-04,0.009724,0.004166,0.000933,0.012742,0.014706
2023-10-05,0.009777,0.000931,0.008325,0.003435,0.015903
2023-10-06,0.011908,0.002542,0.004248,0.005346,0.009135
2023-10-10,-0.004111,-0.004790,-0.002180,-0.008722,-0.004251


,Currency,Sector,Average Daily Volume,Mean Return (%),Daily Std (%)
Ticker,,,,,
AAPL,USD,Technology,4.828203e+07,0.1117,1.7513
ABBV,USD,Healthcare,6.281745e+06,0.1108,1.6273
ABT,USD,Healthcare,1.008423e+07,0.0300,1.4879
ACN,USD,Technology,6.116120e+06,-0.0335,2.1639
AIG,USD,Financial Services,4.324195e+06,0.0552,1.4738
AMZN,USD,Consumer Cyclical,4.652561e+07,0.1160,2.0536
AXP,USD,Financial Services,3.054224e+06,0.1230,1.7637
BA,USD,Industrials,7.140954e+06,0.0303,2.2634
BAC,USD,Financial Services,3.673368e+07,0.1256,1.5943


In [33]:
full_data = pd.DataFrame()

for tick in close_prices:
    full_data[tick] = yf.download(tick, period_start, period_end)["Close"]

full_data["CBIL.TO"] = yf.download("CBIL.TO", period_start, period_end, auto_adjust=True)["Close"]

full_data = full_data

display(full_data)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,CBIL.TO
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,171.301224,132.844528,89.382935,289.540100,56.168282,129.460007,143.918549,187.830002,24.848059,6.02,...,104.262367,73.839996,17.100689,70.040108,146.078232,483.225800,189.679688,128.719467,28.285885,45.243137
2023-10-03,169.970261,132.109756,89.438965,287.633301,54.849430,124.720001,139.653351,188.919998,24.112856,5.98,...,103.408371,70.440002,17.204716,69.378120,143.612396,478.436371,190.746765,127.615593,27.581823,45.229580
2023-10-04,171.212509,132.342728,89.308243,290.601471,55.522896,127.000000,141.174606,186.729996,24.140774,5.86,...,103.089294,72.230003,17.756870,69.195213,144.269958,479.516357,189.333344,127.590660,27.493813,45.243137
2023-10-05,172.444870,132.127686,89.821777,290.855011,56.261822,125.959999,142.199356,186.289993,24.252447,5.20,...,103.042358,71.510002,18.100962,69.997551,141.657974,484.784668,187.807602,127.308487,28.048267,45.247658
2023-10-06,174.988510,132.835602,90.456688,293.231476,56.935284,127.959999,144.355194,187.380005,24.261755,5.15,...,104.055893,73.269997,18.044945,70.359047,143.384064,492.842010,189.511200,128.047180,28.039467,45.256702
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-24,335.664978,263.502045,100.451622,175.998123,74.820000,249.380005,304.717529,196.800003,56.029999,12.27,...,194.259995,205.300003,12.030000,167.495621,269.335388,375.010010,274.070007,90.706924,57.857586,49.906166
2026-09-25,340.811066,262.726807,100.670280,174.708466,74.169998,249.669998,307.937592,198.070007,56.700001,11.57,...,201.970001,201.199997,12.000000,169.511597,276.719360,376.589996,273.790009,92.589050,58.778946,49.906166
2026-09-28,338.143066,264.654968,100.362175,173.081512,72.779999,246.149994,305.385437,184.389999,55.470001,12.51,...,187.479996,204.199997,11.860000,168.429123,276.958191,377.829987,274.100006,92.874825,58.234055,49.906166


Sharpe calculation

In [34]:
half_life = 100

In [35]:
def get_stats(tick, half_life):
    s = full_data[tick].dropna()
    delta = np.log(1 + s.pct_change()).ewm(halflife = half_life)
    standard_deviation = delta.std().iloc[-1]
    average = delta.mean().iloc[-1]

    return standard_deviation, average

In [36]:
Rf = get_stats("CBIL.TO", half_life)[0]
print("Risk free return:", Rf)

sharpe = {}

for tick in close_prices:
    stats = get_stats(tick, half_life)
    sharpe[tick] = (stats[0] - Rf)/stats[1]

sharpe = dict(sorted(sharpe.items(), key = lambda item:item[1]))
print(sharpe)

Risk free return: 0.00015445794031645658
{'PYPL': np.float64(-117.21738988342857), 'CL': np.float64(-101.15083475877516), 'AIG': np.float64(-99.65044620084673), 'LMT': np.float64(-86.07645985603513), 'ACN': np.float64(-44.41514831245113), 'AXP': np.float64(-44.10169389621349), 'UPS': np.float64(-38.994605977107376), 'ABT': np.float64(-32.82247799385365), 'BA': np.float64(-23.41814775623228), 'PEP': np.float64(-16.7590414925412), 'T.TO': np.float64(-7.562648859358836), 'TD.TO': np.float64(7.155567625728956), 'RY.TO': np.float64(8.573920114437547), 'MRK': np.float64(11.108333760412547), 'BB.TO': np.float64(11.90238891993439), 'AAPL': np.float64(13.39659148445312), 'ABBV': np.float64(14.302961780585179), 'KO': np.float64(14.54188950528472), 'BIIB': np.float64(15.875199913951603), 'PFE': np.float64(16.137626385473492), 'C': np.float64(19.943982983042797), 'TXN': np.float64(19.9838862422269), 'CAT': np.float64(20.11819986688738), 'PM': np.float64(20.44403176007088), 'BMY': np.float64(22.238

In [41]:
def weighted_corr(t1, t2):
    d1 = full_data[t1].pct_change(fill_method=None)
    d2 = full_data[t2].pct_change(fill_method=None)
    return d1.ewm(halflife=half_life).corr(d2).dropna().iloc[-1]

Correlation matrix

In [42]:
corr_mat = pd.DataFrame()

for t1 in close_prices:
    for t2 in close_prices:
        corr_mat.loc[t1, t2] = weighted_corr(t1, t2)

In [43]:
display(corr_mat)

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
AAPL,1.000000,0.169149,0.098866,0.180679,0.229776,0.111151,0.246738,0.237700,0.251212,0.022890,...,0.269392,0.214442,0.261143,0.326863,0.223010,0.135810,-0.035881,0.108889,0.215455,0.267525
ABBV,0.169149,1.000000,0.321891,0.187360,0.194110,-0.155674,0.070052,-0.036207,0.075035,-0.015386,...,0.138632,-0.094889,0.016319,0.113246,0.022721,-0.077464,0.135933,0.199028,0.160270,0.138872
ABT,0.098866,0.321891,1.000000,0.285273,0.222522,-0.034579,0.111076,0.081138,0.135495,-0.225666,...,0.169627,-0.137684,0.157454,0.099338,0.000159,-0.144920,0.091165,0.256965,0.141354,0.194552
ACN,0.180679,0.187360,0.285273,1.000000,0.213496,0.088046,0.269787,0.095374,0.107067,0.027500,...,0.301324,-0.026642,0.385755,0.288612,-0.037426,-0.257378,0.074579,0.076987,0.120719,0.213970
AIG,0.229776,0.194110,0.222522,0.213496,1.000000,0.015158,0.185537,0.147174,0.244671,-0.100937,...,0.076436,-0.033823,0.122275,0.052265,0.147040,-0.012408,0.099625,0.288567,0.119157,0.258253
AMZN,0.111151,-0.155674,-0.034579,0.088046,0.015158,1.000000,0.322887,0.273834,0.254508,0.134281,...,0.210283,0.170858,0.226590,-0.286972,0.224261,0.133083,0.038342,0.053068,0.123004,0.255499
AXP,0.246738,0.070052,0.111076,0.269787,0.185537,0.322887,1.000000,0.383087,0.598730,0.158492,...,0.305911,0.202895,0.370173,0.081958,0.429763,0.152980,0.193192,0.180926,0.281189,0.666731
BA,0.237700,-0.036207,0.081138,0.095374,0.147174,0.273834,0.383087,1.000000,0.323247,0.045766,...,0.205021,0.276180,0.184316,0.044927,0.306190,0.263537,0.110285,0.069643,0.235342,0.320841
BAC,0.251212,0.075035,0.135495,0.107067,0.244671,0.254508,0.598730,0.323247,1.000000,0.092415,...,0.165409,0.189872,0.146564,-0.028299,0.505531,0.165843,0.115228,0.207871,0.264538,0.743015
BB.TO,0.022890,-0.015386,-0.225666,0.027500,-0.100937,0.134281,0.158492,0.045766,0.092415,1.000000,...,0.071328,0.218934,0.181671,-0.082448,0.224928,0.205908,0.045081,0.038184,0.156753,0.137061
